# 02 — Modelos supervisionados (Fase 3)
Regressão Logística, Random Forest e XGBoost nos **cenários A–D**, com o **split por linhas**.

- Runtime: **GPU (T4 ou L4)** acelera o XGBoost; os outros modelos rodam em CPU.
- Treino com **todas as fraudes** + fração das legítimas (`train.legit_frac` no `config.yaml`, hoje 20%) e **pesos de classe**.
  A avaliação é feita na **validação completa**. O conjunto de teste só será usado na avaliação final.
- Cada modelo imprime o tempo de treino. Se algum passar de ~15 min, interrompa e me avise.

In [ ]:
import os, sys, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fraud-detection-paysim'
except ImportError:
    PROJECT_DIR = os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

import numpy as np, pandas as pd, joblib
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from fraud.config import load_config
cfg = load_config()
y = cfg['data']['target']
FIG = os.path.join(cfg['paths']['reports_dir'], 'figures'); os.makedirs(FIG, exist_ok=True)
MODELS = cfg['paths']['models_dir']; os.makedirs(MODELS, exist_ok=True)
SCORES = os.path.join(cfg['paths']['reports_dir'], 'scores'); os.makedirs(SCORES, exist_ok=True)
pd.options.display.float_format = '{:,.4f}'.format
print('Projeto:', PROJECT_DIR)

## 1. Dados e split

In [ ]:
from fraud.data import load_parquet, dev_sample
from fraud.split import make_split

df = load_parquet(cfg['paths']['parquet'])
train, val, test = make_split(df, cfg)
print('split:', cfg['split']['mode'])
for nome, p in [('treino', train), ('validação', val), ('teste', test)]:
    print(f"{nome}: {len(p):,} linhas | {int(p[y].sum()):,} fraudes ({100*p[y].mean():.4f}%) | steps {p['step'].min()}–{p['step'].max()}")

train_s = dev_sample(train, cfg['train']['legit_frac'], target=y, seed=cfg['seed'])
n_pos = int(train_s[y].sum()); n_neg = len(train_s) - n_pos
print(f"treino usado: {len(train_s):,} linhas ({n_pos:,} fraudes, {n_neg:,} legítimas)")

## 2. Modelos

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import subprocess

try:
    HAS_GPU = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    HAS_GPU = False
print('GPU disponível:', HAS_GPU)

def make_models():
    return {
        'logreg': make_pipeline(StandardScaler(),
                                LogisticRegression(class_weight='balanced', max_iter=500)),
        'random_forest': RandomForestClassifier(
            n_estimators=150, max_depth=14, min_samples_leaf=5, max_samples=0.5,
            class_weight='balanced_subsample', n_jobs=-1, random_state=cfg['seed']),
        'xgboost': XGBClassifier(
            n_estimators=400, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=n_neg / max(n_pos, 1), tree_method='hist',
            device='cuda' if HAS_GPU else 'cpu', eval_metric='aucpr',
            random_state=cfg['seed']),
    }

## 3. Treino e avaliação na validação (3 modelos × 4 cenários)

In [ ]:
from sklearn.metrics import precision_recall_curve
from fraud.features import SCENARIOS, build_features
from fraud.metrics import evaluate

resultados, curvas, modelos_salvos = [], {}, {}
for letra, variante in SCENARIOS.items():
    Xtr, ytr = build_features(train_s, variante), train_s[y]
    Xva, yva = build_features(val, variante), val[y]
    for nome, modelo in make_models().items():
        t0 = time.time()
        modelo.fit(Xtr, ytr)
        score = modelo.predict_proba(Xva)[:, 1]
        seg = time.time() - t0
        r = evaluate(yva.values, score, amount=val['amount'].values)
        r.update({'cenario': letra, 'modelo': nome, 'treino_seg': round(seg, 1)})
        resultados.append(r)
        curvas[(letra, nome)] = precision_recall_curve(yva.values, score)[:2]
        np.save(os.path.join(SCORES, f'val_{nome}_{letra}.npy'), score.astype('float32'))
        modelos_salvos[(letra, nome)] = modelo
        print(f"ok  cenário {letra} | {nome:14s} | {seg:6.1f}s | PR-AUC {r['pr_auc']:.4f}")

res = pd.DataFrame(resultados).set_index(['cenario', 'modelo'])
res.to_csv(os.path.join(cfg['paths']['reports_dir'], 'supervisionados_validacao.csv'))

## 4. Resultados

In [ ]:
print('PR-AUC (validação)')
display(res['pr_auc'].unstack('modelo'))
print('Recall em VALOR no corte de melhor F1')
display(res['bestF1_value_recall'].unstack('modelo'))
print('Taxa de falsos positivos no corte de melhor F1')
display(res['bestF1_fpr'].unstack('modelo'))
cols = ['prevalencia_%', 'pr_auc', 'recall@precision>=90%', 'bestF1_precision', 'bestF1_recall',
        'bestF1_f1', 'bestF1_fpr', 'bestF1_value_recall', 'treino_seg']
display(res[cols])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
for ax, letra in zip(axes.ravel(), SCENARIOS):
    for nome in ['logreg', 'random_forest', 'xgboost']:
        p, r = curvas[(letra, nome)]
        ax.plot(r, p, label=nome)
    ax.set_title(f'Cenário {letra} — {SCENARIOS[letra]}', fontsize=10)
    ax.set_xlabel('recall'); ax.set_ylabel('precision'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'supervisionados_pr_curves.png'), dpi=130); plt.show()

## 5. O que o XGBoost usa para decidir (cenários B e C)

In [ ]:
for letra in ['B', 'C']:
    m = modelos_salvos[(letra, 'xgboost')]
    imp = pd.Series(m.feature_importances_, index=build_features(val.head(5), SCENARIOS[letra]).columns)
    print(f'Cenário {letra}: top 8 features')
    display(imp.sort_values(ascending=False).head(8))

In [ ]:
# Salva os modelos do cenário B (principal) para as próximas fases
for nome in ['logreg', 'random_forest', 'xgboost']:
    joblib.dump(modelos_salvos[('B', nome)], os.path.join(MODELS, f'{nome}_B.joblib'), compress=3)
print('modelos salvos em', MODELS)

## O que enviar de volta
Copie as saídas das seções **3** (linhas `ok ...`), **4** (as 3 tabelas de PR-AUC / recall em valor / FPR) e **5**.